# Arducam Sensor Explore

There is very little information on the sensors from Arducam. This notebook explores some of the relevant registers and settings of the board.

Note that these observations were made with MT9J001/003 on board UC593 Rev.C and may be not valid on other setups.

In [ ]:
import os, sys, time
from pathlib import Path
print(sys.version_info)

import arducam_config_parser
import ArducamSDK

import numpy as np
import cv2
from IPython.display import Image
import matplotlib.pyplot as plt

handle = None

In [ ]:
# Pick a configuration file with GUI
# %gui tk
# import tkinter
from tkinter.filedialog import askopenfilename
configName = askopenfilename() # show an "Open" dialog box and return the path to the selected file
# %gui
print("Config file:", configName, type(configName), bool(configName))

In [ ]:
## All configuration files for MT9J001/2 from the main repo.
# configName = Path('../Config') / 'MT9J001_MONO_8b_640x480.cfg'
# configName = Path('../Config') / 'MT9J001_MONO_12b_640x480.cfg'
# configName = Path('../Config') / 'MT9J003_RAW_8b_640x480.cfg'
## configName = Path('../Config') / 'MT9J003_RAW_12b_640x480.cfg'  # not in the original set

# configName = Path('../Config') / 'MT9J001_MONO_8b_1280x720.cfg'
# configName = Path('../Config') / 'MT9J001_MONO_12b_1280x720.cfg'
# configName = Path('../Config') / 'MT9J003_RAW_8b_1280x720.cfg'
## configName = Path('../Config') / 'MT9J003_RAW_12b_1280x720.cfg'  # not in the original set

# configName = Path('../Config') / 'MT9J001_MONO_8b_1920x1080.cfg'
# configName = Path('../Config') / 'MT9J001_MONO_12b_1920x1080.cfg'
# configName = Path('../Config') / 'MT9J003_RAW_8b_1920x1080.cfg'
## configName = Path('../Config') / 'MT9J003_RAW_12b_1920x1080.cfg'  # not in the original set

# configName = Path('../Config') / 'MT9J001_MONO_8b_3664x2748.cfg'
# configName = Path('../Config') / 'MT9J001_MONO_12b_3664x2748.cfg'
# configName = Path('../Config') / 'MT9J003_RAW_8b_3664x2748.cfg'    # Error, pll_multiplier set to 0x90, over maximum 0x80
# configName = Path('../Config') / 'MT9J003_RAW_12b_3664x2748.cfg'   # Error, needs active VRCMD = 0xD7, 0x4600, 0x0100, 1, 0x85

# configName = Path('../Config') / 'MT9J001_MONO_8b_3664x512.cfg'
# configName = Path('../Config') / 'MT9J001_MONO_8b_3664x64.cfg'

configName = Path('../Config') / 'test.cfg'


In [ ]:
if handle is not None:
    ArducamSDK.Py_ArduCam_close(handle)

handle = ArducamSDK.initCameraFromFile(configName)

## Image orientation

SMIA compatibility, p.150: `R0x0101`, designated as `image_orientation`. Alias register (Manufacturer Specific): `R0x301D`. These registers are used exclusively for image orientation. Only lowest 2 bits active independently, on and off:

* Bit 0: HMirror
* Bit 1: VFlip

Main register: `R0x3040`, named `read_mode`. Image orientation only uses two bits, while the others contain information on subsampling the image:

* Bit 14: HMirror
* Bit 15: VFlip

Only setting `R0x3040` has effect. The change is not reflected in `R0x0101` or `R0x301D` (they remain at 0). Also, setting these to a non-zero value does not have any effect at all; they remain 0, but `R0x3040` is reset to normal, along with the observed image orientation.

Related: register `R0x0006` (SMIA) and `R0x3024` (Manufacturer-specific) are read-only, for pixel order in the Bayer array. This is stated explicitly in the Register Reference. The order changes with image orientation, and possibly with parity of starting addresses in x and y. However, **the value of these registers does not change and remains 0**. We have to track manually which color is the first pixel read, and implicitly, which debayering algorithm to use. Related to this is the parity of starting addresses in `x` and `y` directions. Usually, these are even numbers, to keep the Bayer order, 
* Normal: `COLOR_BayerGB2BGR`
* HMirror: `COLOR_BayerBG2BGR`
* VFlip: `COLOR_BayerRG2BGR`
* both: `COLOR_BayerGR2BGR`

Setting/resetting these bits should be made on the register value retrieved from `R0x3040` so other values in the same register are not changed. Masks:
```
vflip = (1<<15)
hmirr = (1<< 14)
reg  |= vflip  # turn on
reg  &= vflip  # turn off
```

In [ ]:
# Image orientation in R0x3040
err, reg = ArducamSDK.Py_ArduCam_readSensorReg(handle, 0x3040)
print(err, hex(reg))

# vertical flip
reg |= (1<<15)  # on
# reg &= ~(1<<15)  # off

# Horizontal mirror
reg |= (1<<14)  # on
# reg &= ~(1<<14)  # off

ArducamSDK.Py_ArduCam_writeSensorReg(handle, 0x3040, reg)
err, reg = ArducamSDK.Py_ArduCam_readSensorReg(handle, 0x3040)
print(err, hex(reg))


In [ ]:
# Image orientation, in R0x0101 and R0x301D
# 1: mirror; 2: vflip; 3: both
reg = 3
ArducamSDK.Py_ArduCam_writeSensorReg(handle, 0x0101, reg)
# ArducamSDK.Py_ArduCam_writeSensorReg(handle, 0x301d, reg)

In [ ]:
print("Image orientation registers:")
err, reg = ArducamSDK.Py_ArduCam_readSensorReg(handle, 0x3040)  # Reg 0x3040 is the main (bits 15, 14)
print('Reg 0x3040:', err, hex(reg >> 14), "  Full register:", hex(reg))
# the others are aliases are full 8 bit registers used only for these two bits
err, reg = ArducamSDK.Py_ArduCam_readSensorReg(handle, 0x0101)
print('Reg 0x0101:', err, hex(reg & 0xff))
err, reg = ArducamSDK.Py_ArduCam_readSensorReg(handle, 0x301d)
print('Reg 0x301d:', err, hex(reg & 0xff))

print("Pixel order registers:")
err, reg = ArducamSDK.Py_ArduCam_readSensorReg(handle, 0x301D)
print('Reg 0x301D:', reg & 0xff, hex(reg & 0xff))
err, reg = ArducamSDK.Py_ArduCam_readSensorReg(handle, 0x0006)
print('Reg 0x0006:', reg & 0xff, hex(reg & 0xff))

## Gain registers

Main registers described in Register Reference are in the table below

<table>
    <tr>
        <th>Global</th>
        <th>GreenR<br>Green1</th>
        <th>Red</th>
        <th>Blue</th>
        <th>GreenB<br>Green2</th>
        <th>Notes</th>
    </tr>
    <tr>
        <td>0x0204</td>
        <td>0x0206</td>
        <td>0x0208</td>
        <td>0x020A</td>
        <td>0x020C</td>
        <td>(SMIA) 16-bit register, only lower 8 used. Range: 8-255</td>
    </tr>
    <tr>
        <td>0x3028</td>
        <td>0x302A</td>
        <td>0x302C</td>
        <td>0x302E</td>
        <td>0x3030</td>
        <td>(Manuf.) 16-bit register, only lower 8 used. Range: 8-255</td>
    </tr>
    <tr>
        <td>0x305E</td>
        <td>0x3056</td>
        <td>0x305A</td>
        <td>0x3058</td>
        <td>0x305C</td>
        <td>(Manuf.) 16-bit register, encoding both digital and analog gain<br>
            Analog section is split in individual amplification steps (gain2, gain1, initial)<td>
</table>

Gain value 8 in SMIA registers corresponds to an actual gain of 1, 16 corresponds to 2, and so on. However, for SMIA values not power of 2, the mapping is not linear.

In [ ]:
# Gain, analog; does not translate directly into actual gain
# Format:   0001|222|7 7|bbb bbbb
#     0001: digital gain, 1-7
#      222: analog gain, val [0,4,6] to gain [1,2,4]
#       77: any bit combination would work, for gains 1,2,4; only 00,01,11 recommended
# bbb bbbb: first bit recommended 1; others can be any. Increments of 1/64.

gainRegs0 = [(0x0204, 0x24),  # analog gain global
             (0x0206, 0x34),  # analog gain GreenR
             (0x0208, 0x44),  # analog gain Red
             (0x020A, 0x54),  # analog gain Blue
             (0x020C, 0x64),  # analog gain GreenB
           ]
gainRegs1 = [(0x3028, 0x24),  # analog gain global
             (0x302A, 0xff),  # analog gain GreenR
             (0x302C, 0xff),  # analog gain Red
             (0x302E, 0xff),  # analog gain Blue
             (0x3030, 0xff),  # analog gain GreenB
           ]
gainRegs2 = [(0x3056, 0x24),  # analog gain Green1
             (0x3058, 0xff),  # analog gain Blue
             (0x305A, 0xff),  # analog gain Red
             (0x305C, 0xff),  # analog gain Green2
             (0x305E, 0xff),  # analog gain Global
           ]

for rgset in gainRegs0:
    print('Setting ', hex(rgset[0]), hex(rgset[1]))
    ArducamSDK.Py_ArduCam_writeSensorReg(handle, rgset[0], rgset[1])

print('Reading SMIA gain registers')
for rgset in gainRegs0:
    _, val = ArducamSDK.Py_ArduCam_readSensorReg(handle, rgset[0])
    print(f'{rgset[0]:#06x}: {val:#06x}, {val}')

print('Reading Manuf gain registers')
for rgset in gainRegs1:
    _, val = ArducamSDK.Py_ArduCam_readSensorReg(handle, rgset[0])
    print(f'{rgset[0]:#06x}: {val:#06x}, {val}')

print('Reading Manuf gain registers (piecewise)')
for rgset in gainRegs2:
    _, val = ArducamSDK.Py_ArduCam_readSensorReg(handle, rgset[0])
    print(f'{rgset[0]:#06x}: {val:#06x} -- {val & 0x7f:#x}, {(val>>7) & 0x3:#x}, {(val>>9) & 7:#x}, {(8/(8-((val>>9)&7)))*(1+((val>>8)&1))*(1+((val>>7)&1))*(val & 0x7f)/64:6.3f} ')


In [ ]:
err, reg = ArducamSDK.Py_ArduCam_readSensorReg(handle, 0x3056)
hex(reg), hex(reg & 0x7f), bin((reg>>7) & 0x3), bin((reg>>9) & 7), (8/(8-((reg>>9)&7)))*(1+((reg>>8)&1))*(1+((reg>>7)&1))*(reg & 0x7f)/64

## Capture Image

In [ ]:
ArducamSDK.Py_ArduCam_setMode(handle,ArducamSDK.CONTINUOUS_MODE)
rtn_val = ArducamSDK.Py_ArduCam_beginCaptureImage(handle)
print("Begin Capture: ",hex(rtn_val))
for i in range(5):
    time0 = time.time()
    rtn_val = ArducamSDK.Py_ArduCam_captureImage(handle)
    if rtn_val<255:
        if ArducamSDK.Py_ArduCam_availableImage(handle):
            err_code, data, img_cfg = ArducamSDK.Py_ArduCam_readImage(handle)
            time1 = time.time()
            dt = time1-time0
            print(f"       Image Captured: {err_code:#06X},    time: {dt:9.6f}, fps: {1/dt:.3f}")
            ArducamSDK.Py_ArduCam_del(handle)
            break
    else:
        # an image may be retrieved even if an error code is returnedm but may be incomplete
        if ArducamSDK.Py_ArduCam_availableImage(handle):
            ArducamSDK.Py_ArduCam_del(handle)
        
    print(f"{i:3}    Capture Image: {rtn_val:#06X}    Available images: {ArducamSDK.Py_ArduCam_availableImage(handle)}")

ArducamSDK.Py_ArduCam_flush(handle)
rtn_val = ArducamSDK.Py_ArduCam_endCaptureImage(handle)
print("End Capture: ",hex(rtn_val))


In [ ]:
if len(data) == img_cfg['u32Height']*img_cfg['u32Width']:
    image = np.frombuffer(data, np.uint8).reshape(img_cfg['u32Height'], img_cfg['u32Width'])
else:
    image = (np.frombuffer(data, np.uint16).reshape(img_cfg['u32Height'], img_cfg['u32Width']) >> (img_cfg['u8PixelBits']-8))
    image = image.astype(np.uint8)
    
err_code, rgb_mode = ArducamSDK.Py_ArduCam_readSensorReg(handle, 0x3024)
print(f'{hex(rgb_mode)=}')
# image = cv2.cvtColor(image, cv2.COLOR_BayerRG2BGR)
# image = cv2.cvtColor(image, cv2.COLOR_BayerGR2BGR)
# image = cv2.cvtColor(image, cv2.COLOR_BayerGB2BGR)
# image = cv2.cvtColor(image, cv2.COLOR_BayerBG2BGR)

# Encode the array to PNG and show it directly with IPython
retval, buffer = cv2.imencode('.png', image)
# Image(data=buffer)
Image(data=buffer, width=1000)

In [ ]:
plt.imshow(image[1300:1500,1500:2000], aspect=1)
# plt.axis('off')

In [ ]:
plt.imshow(image, aspect=1)
# plt.axis('off')

In [ ]:
if handle is not None:
    ArducamSDK.Py_ArduCam_close(handle)
handle = None